In [2]:
from typing import TypedDict,Literal
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from langgraph.constants import START, END
from langgraph.graph import StateGraph

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)

# 1. 定义状态
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    content_type:str

# 2. 定义节点
def node_a(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一首关于{state['topic']}的唐诗")])
    return {"poem": response.content}

def node_b(state: OverAllState) -> OverAllState:
    response = model.invoke([HumanMessage(content=f"写一个关于{state['topic']}的笑话")])
    return {"joke": response.content}

def route(state: OverAllState) ->Literal["node_a", "node_b"]:
    if "诗" in state["content_type"]:
        return "node_a"
    else:
        return "node_b"

# 3. 构建图
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_conditional_edges(START,route)
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)

graph = builder.compile()
graph.invoke({"topic": "猫猫", "content_type": "诗"})

{'topic': '猫猫',
 'poem': '《戏咏猫》\n凝眸窥鼠窟，侧耳辨风檐。\n碧眼藏星斗，乌裘裹玉蟾。\n跃如弦上箭，卧似水中蟾。\n夜半惊蝶梦，花阴印雪签。\n\n注：此诗以唐诗风格咏猫，通过“碧眼藏星斗”等意象展现猫的神秘灵动。尾联“花阴印雪签”暗写猫爪印迹，化实为虚，留白生趣。全篇紧扣猫之形神，亦隐喻唐人尚武好猎之风，于闲适处见锋芒。',
 'content_type': '诗'}